In [0]:
from pyspark.sql.functions import *

fact = spark.read.table("subscription_pipeline.silver.opportunity")

min_date = fact.select(min(to_date("start_date"))).collect()[0][0]
max_date = fact.select(max(to_date("end_date"))).collect()[0][0]

date_df = spark.sql(f"""
SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) AS date
""")

dim_date = date_df.select(
    col("date"),
    year("date").alias("year"),
    month("date").alias("month"),
    dayofmonth("date").alias("day"),
    quarter("date").alias("quarter"),
    weekofyear("date").alias("week"),
    date_format("date", "yyyy-MM").alias("year_month"),
    
    # Financial Year (April-March)
    when(month("date") >= 4, year("date")).otherwise(year("date") - 1).alias("financial_year")
)

dim_date.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("subscription_pipeline.gold.dim_date")